# DocFusion: Level 1 — Document Understanding & EDA

This notebook explores the unified dataset (SROIE + CORD + Find-It-Again)
to build intuition about document layouts, field distributions, and fraud patterns.

In [ ]:
import json
import os
import sys
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = str(Path(".").resolve().parent)
sys.path.insert(0, PROJECT_ROOT)

from utils import load_jsonl

train_path = os.path.join(PROJECT_ROOT, "dummy_data", "train", "train.jsonl")
test_path = os.path.join(PROJECT_ROOT, "dummy_data", "test", "test.jsonl")

train_data = load_jsonl(train_path)
test_data = load_jsonl(test_path)

print(f"Training records: {len(train_data)}")
print(f"Test records: {len(test_data)}")
print(f"\nSample train record:\n{json.dumps(train_data[0], indent=2)}")

## 1. Vendor Distribution

In [ ]:
vendors = [r["fields"]["vendor"] for r in train_data]
vendor_counts = Counter(vendors)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(list(vendor_counts.keys()), list(vendor_counts.values()), color="steelblue")
ax.set_xlabel("Count")
ax.set_title("Vendor Frequency (Training Set)")
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "vendor_distribution.png"), dpi=150)
plt.show()

## 2. Price / Total Distribution

In [ ]:
totals = [float(r["fields"]["total"]) for r in train_data]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(totals, bins=15, color="coral", edgecolor="black")
axes[0].set_title("Total Amount Distribution")
axes[0].set_xlabel("Amount")
axes[0].set_ylabel("Frequency")

forged_totals = [float(r["fields"]["total"]) for r in train_data if r["label"]["is_forged"] == 1]
genuine_totals = [float(r["fields"]["total"]) for r in train_data if r["label"]["is_forged"] == 0]

axes[1].hist(genuine_totals, bins=10, alpha=0.7, label="Genuine", color="green")
axes[1].hist(forged_totals, bins=10, alpha=0.7, label="Forged", color="red")
axes[1].set_title("Total by Forgery Status")
axes[1].set_xlabel("Amount")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "total_distribution.png"), dpi=150)
plt.show()

print(f"Total stats: mean={np.mean(totals):.2f}, std={np.std(totals):.2f}, min={min(totals)}, max={max(totals)}")

## 3. Fraud Type Breakdown

In [ ]:
fraud_types = [r["label"]["fraud_type"] for r in train_data]
fraud_counts = Counter(fraud_types)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Forged vs Genuine
is_forged = [r["label"]["is_forged"] for r in train_data]
forged_total = sum(is_forged)
genuine_total = len(is_forged) - forged_total
axes[0].pie([genuine_total, forged_total], labels=["Genuine", "Forged"],
            autopct="%1.0f%%", colors=["#4CAF50", "#f44336"])
axes[0].set_title("Genuine vs Forged")

# Fraud type breakdown
fraud_only = {k: v for k, v in fraud_counts.items() if k != "none"}
if fraud_only:
    axes[1].bar(fraud_only.keys(), fraud_only.values(), color=["#FF9800", "#2196F3", "#9C27B0"])
    axes[1].set_title("Fraud Type Breakdown")
    axes[1].set_ylabel("Count")

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "fraud_breakdown.png"), dpi=150)
plt.show()

print(f"Fraud type counts: {dict(fraud_counts)}")

## 4. Date Analysis

In [ ]:
from datetime import datetime

dates = [datetime.strptime(r["fields"]["date"], "%Y-%m-%d") for r in train_data]
months = [d.month for d in dates]
month_counts = Counter(months)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, 13), [month_counts.get(m, 0) for m in range(1, 13)], color="teal")
ax.set_xlabel("Month")
ax.set_ylabel("Count")
ax.set_title("Transaction Month Distribution")
ax.set_xticks(range(1, 13))
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "date_distribution.png"), dpi=150)
plt.show()

## 5. Summary Statistics

In [ ]:
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Total training records:  {len(train_data)}")
print(f"Total test records:     {len(test_data)}")
print(f"Unique vendors (train): {len(set(vendors))}")
print(f"Forged (train):         {forged_total} ({forged_total/len(train_data)*100:.0f}%)")
print(f"Genuine (train):        {genuine_total} ({genuine_total/len(train_data)*100:.0f}%)")
print(f"Fraud types:            {list(fraud_only.keys())}")
print(f"Total range:            {min(totals):.2f} - {max(totals):.2f}")
print(f"Total mean:             {np.mean(totals):.2f}")
print(f"Total std:              {np.std(totals):.2f}")
print("=" * 60)